# D53 — FULL 90 câu bằng **ReAct + Calculator tool** (Qwen3-4B)

Bản chạy đầy đủ của phương pháp đã kiểm chứng tay **20/20 câu đúng** trong
`KLTN_D53_ReAct_Calculator_1cau.ipynb` (gồm cả 10 câu khó nhất bộ dữ liệu).

Model **không tự tính tay bất kỳ phép nào** — mọi phép tính đều gọi tool
`Calculator` (sympy, chính xác tuyệt đối, **có bộ nhớ biến**) theo vòng lặp
ReAct thật: `Thought → Action → Action Input → Observation → …`

## Điểm khác bản 1 câu: vòng lặp ReAct chạy THEO LÔ

Chạy tuần tự 90 câu sẽ mất hàng giờ. Ở đây mỗi **vòng** gọi vLLM **một lần**
cho tất cả các câu đang hoạt động (vLLM tự batching), rồi chạy Calculator
riêng cho từng câu, rồi generate tiếp. Số vòng ≈ số lượt gọi tool tối đa
(~10-15), nên tổng cộng chỉ ~15 lượt `llm.generate()` thay vì ~900.

Mỗi câu có **bộ nhớ biến riêng** (`MayTinh()` riêng), **ngân sách token
riêng**, và tự thoát khỏi lô khi viết xong `Final Answer`.

## Prompt & backend giống hệt bản 1 câu

Cell 4 (prompt) và phần backend `MayTinh` ở Cell 5 được **trích nguyên văn**
từ notebook 1 câu bằng script `scratch/build_react_full90.py` — không gõ lại,
nên không có nguy cơ lệch giữa 2 bản.

## Trước khi chạy

Upload `plan_solve_prompts_v3_D53.json` thành Kaggle Dataset (slug gợi ý
`d53-full90`), gắn vào notebook, bật GPU. Notebook tự dùng 2 GPU nếu có
(`tensor_parallel_size=2`) để tăng bộ nhớ KV cache — ngữ cảnh ReAct dài hơn
prompt thường nhiều vì tích luỹ cả lịch sử Thought/Action/Observation.

Kết quả: `/kaggle/working/d53_react_calculator_full90.csv`

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DATA_PATH = '/kaggle/input/d53-full90/plan_solve_prompts_v3_D53.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d53_react_calculator_full90.csv'

MODEL = 'Qwen/Qwen3-4B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau (20/20 dung).
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
# MAX_NEW_TOKENS: TONG token sinh ra qua TAT CA cac luot ReAct cua 1 cau.
# MAX_MODEL_LEN 14336: ngu canh ReAct phinh dan vi tich luy lich su
# Thought/Action/Observation - 11264 nhu cac ban cu se tran giua chung.
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

# Dung 2 GPU neu co: gap doi KV cache (ngu canh ReAct dai) va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
print('So cau:', len(records))

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D53_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
'''


# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: given $|z|$ and $|w|$ and a known complex constant $t$, find $|z-w|$ at the moment $|z + i\overline{w} + t|$ reaches its **minimum**.

Follow this method exactly; do not invent a shorter route.

1. **Substitute.** Put $k = i\overline{w}$. Then $|k| = |w|$, because multiplying by $i$ does not change the modulus (if $w = a+bi$ then $\overline{w} = a-bi$ and $k = i(a-bi) = b + ai$, so $|k| = \sqrt{a^2+b^2} = |w|$). The expression becomes $|z + k + t|$.
2. **Apply the three-term triangle inequality.** $|z + k + t| \ge |t| - |z| - |k|$, and by the equality condition above, the minimum is reached exactly when $z$ and $k$ point in the same direction as each other and both point **opposite to $t$**.
3. **Compute $k$.** Since $k$ points opposite to $t$: $k = -\dfrac{|k|}{|t|} \cdot t = -\dfrac{|w|}{|t|} \cdot t$. Everything on the right is known, so this is a direct computation.
4. **Compute $z$.** Same reasoning: $z = -\dfrac{|z|}{|t|} \cdot t$.
5. **Recover $w$ from $k$.** Multiply $k = i\overline{w}$ by $i$: $i k = i^2 \overline{w} = -\overline{w}$, hence $\overline{w} = -ik$. Compute $-ik$, then take its conjugate to get $w$.
6. **Compute $z - w$**, then compute $|z - w|$. That is the answer value.
7. **Match against the four options.** Test the options in order A, B, C, D with an exact equality check, and stop at the first one that returns `True`.
'''


# =====================================================================
# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D53 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT do sympy tra ve
# (da chay kiem chung), khong phai so bia.
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Xét các số phức $z, w$ thỏa mãn $|z|=\dfrac{5}{6}$ và $|w|=\dfrac{11}{6}$. Khi $|z+i\overline{w}- \dfrac{7}{2}+\dfrac{3}{4}i|$ đạt giá trị nhỏ nhất, $|z-w|$ bằng:
A. $\dfrac{\sqrt{321194}}{246}$ | B. $\dfrac{2\sqrt{321194}}{246}$ | C. $\dfrac{8}{3}$ | D. $-\dfrac{8}{3} + \dfrac{\sqrt{205}}{4}$

<think>
Here $t = -\dfrac{7}{2} + \dfrac{3}{4}i$, $|z| = \dfrac{5}{6}$, $|w| = \dfrac{11}{6}$.

Thought: Store t in memory first so I never have to retype it.
Action: Calculator
Action Input: t = -7/2 + 3/4*I
Observation: t = -7/2 + 3*I/4

Thought: I need |t| for the direction formulas.
Action: Calculator
Action Input: mod_t = Abs(t)
Observation: mod_t = sqrt(205)/4

Thought: k points opposite to t, so k = -(|w|/|t|)*t with |w| = 11/6.
Action: Calculator
Action Input: k = -(11/6)/mod_t * t
Observation: k = 77*sqrt(205)/615 - 11*sqrt(205)*I/410

Thought: z points opposite to t as well, so z = -(|z|/|t|)*t with |z| = 5/6.
Action: Calculator
Action Input: z = -(5/6)/mod_t * t
Observation: z = 7*sqrt(205)/123 - sqrt(205)*I/82

Thought: From k = i*conj(w) I get conj(w) = -i*k.
Action: Calculator
Action Input: w_conj = -I*k
Observation: w_conj = -11*sqrt(205)/410 - 77*sqrt(205)*I/615

Thought: Now take the conjugate of that to recover w itself.
Action: Calculator
Action Input: w = conjugate(w_conj)
Observation: w = -11*sqrt(205)/410 + 77*sqrt(205)*I/615

Thought: Compute the difference z - w.
Action: Calculator
Action Input: diff = z - w
Observation: diff = 103*sqrt(205)/1230 - 169*sqrt(205)*I/1230

Thought: The answer is the modulus of that difference.
Action: Calculator
Action Input: result = Abs(diff)
Observation: result = sqrt(321194)/246

Thought: Test option A exactly.
Action: Calculator
Action Input: Eq(result, sqrt(321194)/246)
Observation: True

Option A matches exactly, so I stop here and do not test B, C, D.
</think>
Final Answer: \boxed{A}
'''


# =====================================================================
# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for **every** computation; never compute anything yourself:

[Step 1] Store $t$ in memory, then compute `mod_t = Abs(t)`.
[Step 2] Compute `k = -(|w|/mod_t) * t`, substituting the numeric $|w|$ from the problem.
[Step 3] Compute `z = -(|z|/mod_t) * t`, substituting the numeric $|z|$ from the problem.
[Step 4] Compute `w_conj = -I*k`, then `w = conjugate(w_conj)`.
[Step 5] Compute `diff = z - w`.
[Step 6] Compute `result = Abs(diff)`.
[Step 7] Test the options in order with `Eq(result, <option>)`; one call per option; and stop at the first `True`.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN
# =====================================================================
# Tong quat cho MOI dang toan (khong chi D53): nhan 1 bieu thuc sympy bat
# ky, tra ve gia tri chinh xac tuyet doi. Ho tro:
#   - gan bien:  "ten = bieu_thuc"  -> luu vao bo nho, dung lai o luot sau
#     (xoa han lo hoi model chep tay lai so dai - nguon loi lon nhat)
#   - so khop :  "Eq(a, b)"         -> True/False chinh xac, khong xap xi
#   - moi phep cong tru nhan chia phan so, can thuc, so phuc, mo dun, lien hop
import sympy as sp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            parsed = sp.sympify(s, locals=self.ns)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri = sp.expand(sp.simplify(parsed))
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# =====================================================================
# Moi vong: gom TAT CA cac cau dang hoat dong -> goi llm.generate() 1 LAN
# (vLLM tu batching) -> chay Calculator rieng cho tung cau -> chen
# Observation that -> vong sau. Cau nao viet xong "Final Answer" thi roi
# khoi lo. Nho vay ~15 luot generate() thay vi ~900 luot tuan tu.
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 25
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh()          # bo nho bien RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        # Luot chot (da het han muc goi tool): BO stop de model buoc phai
        # viet Final Answer thay vi goi tool tiep.
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:                    # vua chay luot chot -> ket thuc
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:      # model tu ket thuc / het token
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        # --- model vua goi tool: chay that va chen Observation that ---
        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm                      # lay Action Input CUOI CUNG
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print('*** TAT CA 90 CAU DEU DUNG ***')
print()
print('Da luu:', OUT_PATH)